# Orchestration

Generated from the book sources. Do not edit by hand: changes belong in the `.qmd` chapter.

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()

client = OpenAI()

CHAT_MODEL = os.environ["CHAT_MODEL"]

RETURN_POLICY = [
    {"id": "§2.1", "text": "Standard items may be returned within "
     "14 days of delivery."},
    {"id": "§4.2", "text": "Refunds are issued to the original "
     "payment method within 5 business days after the returned item "
     "has been received."},
    {"id": "§4.3", "text": "Discounted items are excluded from "
     "refunds but may be exchanged within 14 days."},
]

def tokens(text: str):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def search_return_policy(query: str):
    words = tokens(query)
    ranked = sorted(
        RETURN_POLICY,
        key=lambda p: len(words & tokens(p["text"])),
        reverse=True,
    )
    return {"passages": ranked[:2]}

ORDERS = {
    "A-1001": {"refund_issued": True, "refund_date": "2026-06-24",
               "amount_eur": 79.90},
    "A-1002": {"refund_issued": False,
               "reason": "returned item not yet received"},
}

def get_refund_status(order_id: str):
    if order_id in ORDERS:
        return ORDERS[order_id]
    return {"error": f"unknown order id '{order_id}'"}

policy_tool = {
  "type": "function",
  "function": {
    "name": "search_return_policy",
    "description": "Search the return policy for relevant passages.",
    "parameters": {
      "type": "object",
      "properties": {"query": {"type": "string"}},
      "required": ["query"],
      "additionalProperties": False,
    },
  },
}

refund_tool = {
  "type": "function",
  "function": {
    "name": "get_refund_status",
    "description": "Look up the refund status of an order.",
    "parameters": {
      "type": "object",
      "properties": {"order_id": {"type": "string"}},
      "required": ["order_id"],
      "additionalProperties": False,
    },
  },
}

### A generic single-tool worker loop and the two specialist wrappers

`lst-worker-agents`

In [ ]:
def run_worker(system_prompt, tool_schema, implementation, request,
               max_steps=3):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": request},
    ]
    for _ in range(max_steps):
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=[tool_schema],
            tool_choice="auto",
            temperature=0.0,
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(
            {"role": "assistant", "tool_calls": msg.tool_calls}
        )
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(implementation(**args)),
            })
    return "The worker could not complete the request."

def ask_policy_agent(request: str):
    print(f"  [policy agent] {request}")
    return {"answer": run_worker(
        "You answer questions about the return policy. Ground every "
        "claim in retrieved passages and cite section ids.",
        policy_tool, search_return_policy, request)}

def ask_orders_agent(request: str):
    print(f"  [orders agent] {request}")
    return {"answer": run_worker(
        "You answer questions about order and refund status using "
        "the order database.",
        refund_tool, get_refund_status, request)}

### Uniform delegation schemas exposing the workers as tools to the supervisor

`lst-delegation-schemas`

In [ ]:
def delegation_schema(name, description):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {"request": {"type": "string"}},
                "required": ["request"],
                "additionalProperties": False,
            },
        },
    }

worker_schemas = [
    delegation_schema(
        "ask_policy_agent",
        "Delegate a question about the return policy to the "
        "policy specialist."),
    delegation_schema(
        "ask_orders_agent",
        "Delegate a question about an order or refund status to "
        "the orders specialist."),
]

WORKERS = {
    "ask_policy_agent": ask_policy_agent,
    "ask_orders_agent": ask_orders_agent,
}

### The supervisor loop: the same loop shape, with agents as its action space

`lst-run-supervisor`

In [ ]:
def run_supervisor(user_input: str, max_steps: int = 6):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer support supervisor. Delegate "
                "each part of the request to the matching specialist "
                "agent, then combine their answers into one response."
            ),
        },
        {"role": "user", "content": user_input},
    ]
    for _ in range(max_steps):
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=worker_schemas,
            tool_choice="auto",
            temperature=0.0,
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(
            {"role": "assistant", "tool_calls": msg.tool_calls}
        )
        for tc in msg.tool_calls:
            worker = WORKERS[tc.function.name]
            args = json.loads(tc.function.arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(worker(**args)),
            })
    return "Escalation: supervisor budget exhausted."

In [ ]:
answer = run_supervisor(
    "Has the refund for order A-1001 been issued? And how long do "
    "I have to return another item from the same delivery?"
)
print(answer)

### State schema and node functions for the LangGraph retrieval graph

`lst-langgraph-nodes`

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class SupportState(TypedDict):
    question: str
    passages: list
    answer: str

def retrieve(state: SupportState):
    result = search_return_policy(state["question"])
    return {"passages": result["passages"]}

def generate(state: SupportState):
    context = "\n".join(
        f"- {p['id']}: {p['text']}" for p in state["passages"]
    )
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": (
                "Answer using only the provided policy passages. "
                "Cite section ids.")},
            {"role": "user", "content": (
                f"Passages:\n{context}\n\n"
                f"Question: {state['question']}")},
        ],
    )
    return {"answer": resp.choices[0].message.content}

### Wiring, compiling, and invoking the graph

`lst-langgraph-compile`

In [ ]:
graph = StateGraph(SupportState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
app = graph.compile()

final_state = app.invoke(
    {"question": "How long do I have to send back an item?"}
)
print(final_state["answer"])